In [ ]:
import torch
from torch_geometric.data import Data
import pandas as pd

In [ ]:
df = pd.read_csv("data/processed_data.csv")
edges = pd.read_csv("data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv")

In [ ]:
X = df.drop(columns=["class", "txId"]).values
y = df["class"].values

x = torch.tensor(X, dtype=torch.float)
y = torch.tensor(y, dtype=torch.long)

In [ ]:
edge_index = torch.tensor(edges.values.T, dtype=torch.long)

In [ ]:
data = Data(x=x, edge_index=edge_index, y=y)

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

In [ ]:
model = GCN(x.shape[1], 64, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(20):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.nll_loss(out, data.y)
    loss.backward()
    optimizer.step()
    print(loss.item())

In [ ]:
model.eval()
pred = model(data.x, data.edge_index).argmax(dim=1)

from sklearn.metrics import classification_report
print(classification_report(y, pred))